# PROJECT 2 : FIRST NEW VERSION OF SPR FUNCTION
This consist of using OLS + VAR (1) to improove the results of the interpolation function

## Import libraries

In [1]:
import numpy as np           # For numerical computing
import pandas as pd          # For data manipulation and analysis
import matplotlib.pyplot as plt  # For plotting and visualization
import scipy                 # For scientific computing
import sklearn               # For machine learning
import geopandas as gpd      # For geospatial data
import rasterio              # For raster data processing
import xarray                # For working with labeled multi-dimensional arrays
import tensorflow as tf      # For deep learning and neural networks
import torch                 # For deep learning with PyTorch
import seaborn as sns        # For statistical data visualization
import plotly.express as px  # For interactive plots
import statsmodels.api as sm  # For statistical modeling
import folium               # For interactive maps
import networkx as nx        # For complex networks
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import random
import os
from sklearn.utils import resample
import joblib  # for saving data
from multiprocessing import Pool
from collections import Counter
from concurrent.futures import ProcessPoolExecutor
import multiprocessing
from joblib import Parallel, delayed
from tqdm import tqdm
from pykrige.ok import OrdinaryKriging
from skgstat import Variogram
from pykrige.uk import UniversalKriging
from pykrige.rk import Krige
from shapely.geometry import Point
import geopandas as gpd
import warnings
from pyproj import CRS, Transformer
from pandas.plotting import autocorrelation_plot
from statsmodels.graphics.tsaplots import plot_acf

2026-04-24 13:04:27.717630: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-24 13:04:27.742370: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-24 13:04:28.208396: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


## Auto saving my script every 60 seconds

In [2]:
%autosave 60

Autosaving every 60 seconds


# Loading data

In [3]:
data_coord = pd.read_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data/data_coord.csv")
data_pr = pd.read_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data/data_pr.csv")
data_tasmin = pd.read_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data/data_tasmin.csv")
data_tasmax = pd.read_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data/data_tasmax.csv")
# clim_pr = pd.read_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data/clim_pr.csv")
# clim_tasmin = pd.read_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data/clim_tasmin.csv")
# clim_tasmax = pd.read_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data/clim_tasmax.csv")

# Defining period and indexes

In [4]:
# Extract longitude and latitude columns
rlon = data_coord["lon"]
rlat = data_coord["lat"]

# Define region A
region_A1 = data_coord[(rlat >= -2) & (rlat <= 4) & (rlon >= 13) & (rlon <= 19)].index
region_A2 = data_coord[(rlat >= -1) & (rlat <= 3) & (rlon >= 14) & (rlon <= 18)].index
region_A3 = data_coord[(rlat >= 0) & (rlat <= 2) & (rlon >= 15) & (rlon <= 17)].index

# Define region B
region_B1 = data_coord[(rlat >= 7) & (rlat <= 13) & (rlon >= 14) & (rlon <= 20)].index
region_B2 = data_coord[(rlat >= 8) & (rlat <= 12) & (rlon >= 15) & (rlon <= 19)].index
region_B3 = data_coord[(rlat >= 9) & (rlat <= 11) & (rlon >= 16) & (rlon <= 18)].index

# Defining the periods (1-based indexing in R becomes 0-based in Python)
rcm_period = np.arange(0, 10950)         # 0 to 10949 → 30 years
obs_period_1 = np.arange(7300, 10950)    # 7301:10950 → 7300 to 10949
obs_period_2 = np.arange(10950, 14600)   # 10951:14600 → 10950 to 14599
obs_period_3 = np.arange(14600, 18250)   # 14601:18250 → 14600 to 18249

# Periode de validation et de test
obs_period_hp = np.arange(7300, 9125)
obs_period_test = np.arange(9125, 10950)

## Choose the period and the region
obs_period = obs_period_test # choose the period
region = region_A1 # choose the region
n_years_obs = 5 # number of years in the observation period
n_years_ano = 30 # number of years in the anomaly period

## Define the seed : Set the random seed for reproducibility
my_seed = 2244677 # seed for reproducibility

## Function to calculate anomalies : mensual climatologies with monthly daily means

In [5]:
def monthly_climatology(df):
    """
    Compute monthly climatology from a daily-timeseries DataFrame.
    Assumes non-leap years (365 days). Uses full years only.
    1. Computes the mean for each day of the month across all years.
    2. Then averages these daily means for each month to get the monthly mean.

    Parameters
    ----------
    df : pandas.DataFrame
        Daily data with shape (Njours_total, Nstations)

    Returns
    -------
    climatologie : pandas.DataFrame
        Monthly climatology (12 x Nstations) indexed by month names.
    """
    Njours_total, Nstations = df.shape
    Nyears = Njours_total // 365
    if Nyears < 1:
        raise ValueError("Input must contain at least 365 days to compute climatology.")

    # Reshape to (Nyears, 365, Nstations)
    data_reshaped = df.to_numpy().reshape(Nyears, 365, Nstations)

    # Days per month for a non-leap year and month names
    days_by_month = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    month_names = ["January", "February", "March", "April", "May", "June",
        "July", "August", "September", "October", "November", "December"]

    monthly_means = {}
    start = 0
    for name, ndays in zip(month_names, days_by_month):
        day_indices = list(range(start, start + ndays))
        start += ndays
        # data_month: (Nyears, ndays, Nstations) : select days and data for the month
        data_month = data_reshaped[:, day_indices, :]
        # mean over days by month for each year -> (Nyears, Nstations)
        daily_mean_per_year = data_month.mean(axis=1) # (Nyears, Nstations)
        # mean over years -> (Nstations,)
        monthly_mean = daily_mean_per_year.mean(axis=0) # (Nstations,)
        monthly_means[name] = monthly_mean # Store the monthly mean

    climatologie = pd.DataFrame.from_dict(monthly_means, orient="index", columns=df.columns) # (12, Nstations)
    return climatologie

# Data preparation : defining data on the period and region considered

In [6]:
# Assuming data.pr, data.tasmin, data.tasmax are numpy arrays or pandas DataFrames
# Define these with actual data or load from files
# Example placeholders (replace these with real data):
# data_pr, data_tasmin, data_tasmax = ...

# Example: data should be NumPy arrays or similar
# Using .loc because region_colnames are column names
# rcm_period and obs_period are row positions and region_colnames are column names
## RCM data
region_colnames = [f"V{col}" for col in region]
data_pr_grid = data_pr.loc[rcm_period, region_colnames]
data_pr_grid[data_pr_grid < 0.5] = 0  # remove noise
data_tasmin_grid = data_tasmin.loc[rcm_period, region_colnames]
data_tasmax_grid = data_tasmax.loc[rcm_period, region_colnames]

## Climatology data
## Climatology for precipitation, tmin and tmax
climatology_pr = monthly_climatology(data_pr_grid)
climatology_tasmin = monthly_climatology(data_tasmin_grid)
climatology_tasmax = monthly_climatology(data_tasmax_grid)
# climatology_pr = clim_pr.loc[:, region_colnames]
# climatology_tasmin = clim_tasmin.loc[:, region_colnames]
# climatology_tasmax = clim_tasmax.loc[:, region_colnames]

## Observed data
data_pr_obs = data_pr.loc[obs_period, region_colnames]
data_pr_obs[data_pr_obs < 0.5] = 0  # remove noise
data_tasmin_obs = data_tasmin.loc[obs_period, region_colnames]
data_tasmax_obs = data_tasmax.loc[obs_period, region_colnames]

## Coordinate data for KED interpolation
data_coord_grid = data_coord.loc[region, :]

## Original data for comparison
data_pr_obs_orig = data_pr.loc[obs_period, region_colnames]
data_pr_obs_orig[data_pr_obs_orig < 0.5] = 0  # remove noise
data_tasmin_obs_orig = data_tasmin.loc[obs_period, region_colnames]
data_tasmax_obs_orig = data_tasmax.loc[obs_period, region_colnames]

# Calculate anomalies

In [7]:
def climatology_fun(climatology, nyears):
    # Daily distribution per month (non-leap year)
    days_by_month = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    month_index = np.repeat(np.arange(12), days_by_month)  # 0-based month indices
    climatology_year = climatology.iloc[month_index, :]         # Shape: (365, n_cols)
    
    # Repeat for the number of years
    climatology_nyears = np.tile(climatology_year, (nyears, 1))  # Shape: (365 * nyears, n_cols)
    return climatology_nyears

# For grid data (30 years)
anomaly_pr_grid = data_pr_grid / climatology_fun(climatology_pr, nyears=n_years_ano) - 1 # multiplicative for precipitation. centered
anomaly_tasmin_grid = data_tasmin_grid - climatology_fun(climatology_tasmin, nyears=n_years_ano) # additive for temperature
anomaly_tasmax_grid = data_tasmax_grid - climatology_fun(climatology_tasmax, nyears=n_years_ano) # additive for temperature

# For observations (10 years)
anomaly_pr_obs = data_pr_obs / climatology_fun(climatology_pr, nyears=n_years_obs) - 1# multiplicative for precipitation. centered
anomaly_tasmin_obs = data_tasmin_obs - climatology_fun(climatology_tasmin, nyears=n_years_obs) # additive for temperature
anomaly_tasmax_obs = data_tasmax_obs - climatology_fun(climatology_tasmax, nyears=n_years_obs) # additive for temperature

# A copy of original anomalies for comparison
anomaly_pr_obs_orig = anomaly_pr_obs.copy()
anomaly_tasmin_obs_orig = anomaly_tasmin_obs.copy()
anomaly_tasmax_obs_orig = anomaly_tasmax_obs.copy()

## Scale climatology for GRID and OBS on the same shape as anomalies

In [8]:
# For RCM data (30 years)
clim_pr_grid = climatology_fun(climatology_pr, nyears=n_years_ano)
clim_tasmin_grid = climatology_fun(climatology_tasmin, nyears=n_years_ano)
clim_tasmax_grid = climatology_fun(climatology_tasmax, nyears=n_years_ano)

# For observed data (10 years)
clim_pr_obs = climatology_fun(climatology_pr, nyears=n_years_obs)
clim_tasmin_obs = climatology_fun(climatology_tasmin, nyears=n_years_obs)
clim_tasmax_obs = climatology_fun(climatology_tasmax, nyears=n_years_obs)

# ORiginal anomaly data for comparison
anomaly_pr_obs_orig = data_pr_obs_orig / clim_pr_obs - 1 # multiplicative for precipitation
anomaly_tasmin_obs_orig = data_tasmin_obs_orig - clim_tasmin_obs # additive for temperature
anomaly_tasmax_obs_orig = data_tasmax_obs_orig - clim_tasmax_obs # additive for temperature

## Interpolation with 90% NA : Densité de 10%

# Train and test set

In [9]:
# --- Custom rounding function (round .5 up) ---
perc = 0.9  # % of columns to set as test
def custom_round(x):
    return int(np.ceil(x) if x - int(x) == 0.5 else round(x))

# Select test indexes
np.random.seed(my_seed)
random.seed(my_seed)
n_days, n_cols = data_pr_obs.shape
n_test = custom_round(perc * n_cols)
test_indexes = np.random.choice(n_cols, size=n_test, replace=False)
test_indexes.sort()  # Sort for easier reading
train_indexes = [i for i in range(n_cols) if i not in test_indexes]

# --- Set test indices to NaN in all datasets ---
data_pr_obs.iloc[:, test_indexes] = np.nan
data_tasmin_obs.iloc[:, test_indexes] = np.nan
data_tasmax_obs.iloc[:, test_indexes] = np.nan
anomaly_pr_obs.iloc[:, test_indexes] = np.nan
anomaly_tasmin_obs.iloc[:, test_indexes] = np.nan
anomaly_tasmax_obs.iloc[:, test_indexes] = np.nan

## Convert all data to nympy

In [10]:
# Convert all data to numpy
## Precipitation
data_pr_obs_np = data_pr_obs.values if isinstance(data_pr_obs, pd.DataFrame) else data_pr_obs
data_pr_obs_orig_np = data_pr_obs_orig.values if isinstance(data_pr_obs_orig, pd.DataFrame) else data_pr_obs_orig
anomaly_pr_grid_np = anomaly_pr_grid.values if isinstance(anomaly_pr_grid, pd.DataFrame) else anomaly_pr_grid
anomaly_pr_obs_np = anomaly_pr_obs.values if isinstance(anomaly_pr_obs, pd.DataFrame) else anomaly_pr_obs
clim_pr_grid_np = clim_pr_grid.values if isinstance(clim_pr_grid, pd.DataFrame) else clim_pr_grid
clim_pr_obs_np = clim_pr_obs.values if isinstance(clim_pr_obs, pd.DataFrame) else clim_pr_obs

## Tmin
data_tasmin_obs_np = data_tasmin_obs.values if isinstance(data_tasmin_obs, pd.DataFrame) else data_tasmin_obs
data_tasmin_obs_orig_np = data_tasmin_obs_orig.values if isinstance(data_tasmin_obs_orig, pd.DataFrame) else data_tasmin_obs_orig
anomaly_tasmin_grid_np = anomaly_tasmin_grid.values if isinstance(anomaly_tasmin_grid, pd.DataFrame) else anomaly_tasmin_grid
anomaly_tasmin_obs_np = anomaly_tasmin_obs.values if isinstance(anomaly_tasmin_obs, pd.DataFrame) else anomaly_tasmin_obs
clim_tasmin_grid_np = clim_tasmin_grid.values if isinstance(clim_tasmin_grid, pd.DataFrame) else clim_tasmin_grid
clim_tasmin_obs_np = clim_tasmin_obs.values if isinstance(clim_tasmin_obs, pd.DataFrame) else clim_tasmin_obs

## Tmax
data_tasmax_obs_np = data_tasmax_obs.values if isinstance(data_tasmax_obs, pd.DataFrame) else data_tasmax_obs
data_tasmax_obs_orig_np = data_tasmax_obs_orig.values if isinstance(data_tasmax_obs_orig, pd.DataFrame) else data_tasmax_obs_orig
anomaly_tasmax_grid_np = anomaly_tasmax_grid.values if isinstance(anomaly_tasmax_grid, pd.DataFrame) else anomaly_tasmax_grid
anomaly_tasmax_obs_np = anomaly_tasmax_obs.values if isinstance(anomaly_tasmax_obs, pd.DataFrame) else anomaly_tasmax_obs
clim_tasmax_grid_np = clim_tasmax_grid.values if isinstance(clim_tasmax_grid, pd.DataFrame) else clim_tasmax_grid
clim_tasmax_obs_np = clim_tasmax_obs.values if isinstance(clim_tasmax_obs, pd.DataFrame) else clim_tasmax_obs

# ## Original data
# data_pr_obs_orig_np = data_pr_obs_orig.values if isinstance(data_pr_obs_orig, pd.DataFrame) else data_pr_obs_orig
# data_tasmin_obs_orig_np = data_tasmin_obs_orig.values if isinstance(data_tasmin_obs_orig, pd.DataFrame) else data_tasmin_obs_orig
# data_tasmax_obs_orig_np = data_tasmax_obs_orig.values if isinstance(data_tasmax_obs_orig, pd.DataFrame) else data_tasmax_obs_orig

# Original anomaly data for comparison
anomaly_pr_obs_orig_np = anomaly_pr_obs_orig.values if isinstance(anomaly_pr_obs_orig, pd.DataFrame) else anomaly_pr_obs_orig
anomaly_tasmin_obs_orig_np = anomaly_tasmin_obs_orig.values if isinstance(anomaly_tasmin_obs_orig, pd.DataFrame) else anomaly_tasmin_obs_orig
anomaly_tasmax_obs_orig_np = anomaly_tasmax_obs_orig.values if isinstance(anomaly_tasmax_obs_orig, pd.DataFrame) else anomaly_tasmax_obs_orig

# SPR anonmalies with VAR(1)

In [11]:
# ============================================================
# TEMPERATURE
# ============================================================
def spr_interp_anomaly_temp_var_em(anomaly_rcm_grid, anomaly_obs_grid, climatology_obs_grid,
                                    n_spatterns=10, n_jobs=-1, lambda_var=1e-10,
                                    max_iter=20, tol=1e-4, init_A='zero',
                                    ridge_var=False, gamma_ridge_var=0, 
                                    ridge_betas=False, gamma_ridge_betas=0,
                                    verbose=True):
    """
    SPR interpolation with EM optimization starting from A = 0.
    lambda_var = 0 correspond to no VAR(1) smoothing
    Algorithm:
    ----------
    1. Initialize A = 0 (or A = I)
    2. Step I: Estimate weights B with fixed A via regularized least squares
    3. Step II: Estimate transition matrix A with fixed B via OLS regression
    4. Repeat until convergence
    
    Parameters
    ----------
    init_A : str, {'zero', 'identity'}
        Initialization for A matrix
    max_iter : int
        Maximum EM iterations
    tol : float
        Convergence tolerance on cost function
    
    Returns
    -------
    data_interp_spr : np.ndarray (T, D)
    B : np.ndarray (T, K) - optimized weights
    A_est : np.ndarray (K, K) - optimized transition matrix
    costs : list - cost function history
    """
    
    # Convert inputs
    anomaly_rcm_grid = anomaly_rcm_grid.values if isinstance(anomaly_rcm_grid, pd.DataFrame) else anomaly_rcm_grid
    anomaly_obs_grid = anomaly_obs_grid.values if isinstance(anomaly_obs_grid, pd.DataFrame) else anomaly_obs_grid
    climatology_obs_grid = climatology_obs_grid.values if isinstance(climatology_obs_grid, pd.DataFrame) else climatology_obs_grid
    
    # Step 1: SVD decomposition
    svd_rcm = TruncatedSVD(n_components=n_spatterns, random_state=0)
    svd_rcm.fit(anomaly_rcm_grid)
    spatterns = svd_rcm.components_.T  # (D_full, K)
    
    # Step 2: Remove missing columns
    first_day = anomaly_obs_grid[0, :]
    indexes_NA_col = np.where(np.isnan(first_day))[0]
    data_valid = np.delete(anomaly_obs_grid, indexes_NA_col, axis=1)  # (T, D_obs)
    spatterns_model = np.delete(spatterns, indexes_NA_col, axis=0)  # (D_obs, K)
    
    T = anomaly_obs_grid.shape[0]
    K = n_spatterns
    V = spatterns_model  # shorthand
    
    # Check if VAR component is needed
    use_var = lambda_var >= 1e-8 # Ou prendre une valeur encore plus petite pour être sûr de ne pas activer le VAR ?
    if verbose:
        if use_var:
            print(f"Using VAR(1) temporal component (λ={lambda_var:.6f})")
        else:
            print(f"λ={lambda_var:.6f} ≈ 0 → No VAR component (independent temporal estimation)")
    
    # ============================================================
    # INITIALIZATION: A = 0 or A = I
    # ============================================================
    if use_var:
        if init_A == 'zero':
            A_est = np.zeros((K, K))
        elif init_A == 'identity':
            A_est = np.eye(K)
        else:
            raise ValueError("init_A must be 'zero' or 'identity'")
        
        if verbose:
            print(f"Initialized A with {init_A} matrix")
    else:
        # No VAR: A is not needed
        A_est = np.zeros((K, K))
        rho = 0.0
    
    # ============================================================
    # COST FUNCTION
    # ============================================================
    def compute_cost(B_mat, A_mat, data, V_mat, lam):
        """
        J(B, A) = ||Y - VB^T||^2_F + lambda * ||B_{2:T} - A B_{1:T-1}||^2_F
        """
        # Reconstruction error: sum_t ||y_t - V*beta_t||^2
        recon_error = 0.0
        for t in range(T):
            y_t = data[t, :]
            y_hat_t = V_mat @ B_mat[t, :]
            recon_error += np.sum((y_t - y_hat_t) ** 2)
        
        # Temporal penalty: sum_{t=2}^T ||beta_t - A*beta_{t-1}||^2
        temp_penalty = 0.0
        if lam >= 1e-8: # Seuil pour éviter de calculer la pénalité si lambda est très petit (pas de VAR)
            for t in range(1, T):
                residual = B_mat[t, :] - A_mat @ B_mat[t-1, :]
                temp_penalty += np.sum(residual ** 2)
        
        return recon_error + lam * temp_penalty
    
    # ============================================================
    # EM ITERATIONS
    # ============================================================
    costs = []
    B = None  # Will be initialized in first iteration
    
    for iteration in range(max_iter):
        # --------------------------------------------------------
        # STEP I: Estimate weights B with fixed A
        # --------------------------------------------------------
        # Each time step is an independent quadratic problem:
        # beta_t = argmin ||y_t - V*beta||^2 + lambda ||beta - A*beta_{t-1}||^2
        
        B_new = np.zeros((T, K))
        
        # First time step (no temporal prior)
        y_0 = data_valid[0, :]
        if ridge_betas:
            G_0 = V.T @ V + gamma_ridge_betas * np.eye(K)
        else:
            G_0 = V.T @ V + 1e-4 * np.eye(K)  # numerical stability
        rhs_0 = V.T @ y_0
        B_new[0, :] = np.linalg.solve(G_0, rhs_0)
        
        # Subsequent time steps (with temporal regularization if VAR is used)
        for t in range(1, T): # range 1 to T-1
            y_t = data_valid[t, :]
            
            if use_var:
                # Solve: (V^T V + lambda*I) beta_t = V^T y_t + lambda*A*beta_{t-1}
                if ridge_betas:
                    G = V.T @ V + lambda_var * np.eye(K) + gamma_ridge_betas * np.eye(K)
                else:
                    G = V.T @ V + lambda_var * np.eye(K) + 1e-4 * np.eye(K)  # numerical stability
                
                rhs = V.T @ y_t + lambda_var * (A_est @ B_new[t-1, :])
            else:
                # No VAR: each time step is independent
                if ridge_betas:
                    G = V.T @ V + gamma_ridge_betas * np.eye(K)
                else:
                    G = V.T @ V + 1e-4 * np.eye(K) # numerical stability
                rhs = V.T @ y_t
            
            B_new[t, :] = np.linalg.solve(G, rhs)
        
        B = B_new
        
        # --------------------------------------------------------
        # STEP II: Estimate transition matrix A with fixed B
        # --------------------------------------------------------
        if use_var:            
            B_past = B[:-1, :]      # B_{1:T-1}, shape (T-1, K)
            B_future = B[1:, :]     # B_{2:T}, shape (T-1, K)
            
            # Compute A via OLS regression (each row k of A independently)            
            Gram = B_past.T @ B_past  # (K, K)
            
            if ridge_var:
                Gram_reg = Gram + gamma_ridge_var * np.eye(K)
            else:
                Gram_reg = Gram + 1e-4 * np.eye(K)  # numerical stability
            
            # Moore-Penrose pseudo-inverse (or regular inverse if full rank)
            try:
                Gram_inv = np.linalg.inv(Gram_reg)
            except np.linalg.LinAlgError:
                Gram_inv = np.linalg.pinv(Gram_reg)
            
            A_est = B_future.T @ B_past @ Gram_inv  # (K, K)
            
            # Check for NaN/Inf in A_est before computing eigenvalues
            if not np.isfinite(A_est).all():
                raise ValueError(
                    f"A_est contains NaN or Inf values. This typically indicates numerical instability. "
                    f"Try increasing lambda_var (current: {lambda_var}) or reducing n_spatterns (current: {n_spatterns})."
                )
            
            # Apply stability constraint: spectral radius < 1
            rho = np.max(np.abs(np.linalg.eigvals(A_est)))
            if rho > 1:
                A_est = A_est / (rho + 1e-4)  # scale down
                if verbose and iteration == 0:
                    print(f"  Applied spectral radius normalization (ρ={rho:.4f})")
        # else: A_est and rho already set above (no update needed)
        
        # --------------------------------------------------------
        # CHECK CONVERGENCE
        # --------------------------------------------------------
        current_cost = compute_cost(B, A_est, data_valid, V, lambda_var)
        costs.append(current_cost)
        if iteration > 0:
            cost_change = abs(current_cost - prev_cost) / (prev_cost + 1e-4)  # relative change
            if verbose:
                if use_var:
                    print(f"Iter {iteration+1}/{max_iter}: Cost = {current_cost:.6f}, "
                          f"ΔCost = {cost_change:.6e}, ||A||_F = {np.linalg.norm(A_est, 'fro'):.4f}, "
                          f"ρ(A) = {rho:.4f}")
                else:
                    print(f"Iter {iteration+1}/{max_iter}: Cost = {current_cost:.6f}, "
                          f"ΔCost = {cost_change:.6e}")
            if cost_change < tol:
                if verbose:
                    print(f"✓ Converged after {iteration+1} iterations")
                break
        else:
            if verbose:
                if use_var:
                    print(f"Iter {iteration+1}/{max_iter}: Cost = {current_cost:.6f}, "
                          f"||A||_F = {np.linalg.norm(A_est, 'fro'):.4f}")
                else:
                    print(f"Iter {iteration+1}/{max_iter}: Cost = {current_cost:.6f}")
        prev_cost = current_cost
    
    # ============================================================
    # RECONSTRUCTION
    # ============================================================
    def reconstruct_day(t):
        beta_t = B[t, :].reshape(-1, 1)
        anomaly_reconstructed = spatterns @ beta_t
        rec = anomaly_reconstructed + climatology_obs_grid[t, :].reshape(-1, 1)
        return rec.ravel()
    
    interpolated_days = Parallel(n_jobs=n_jobs)(
        delayed(reconstruct_day)(t) for t in tqdm(range(T), desc="Reconstructing")
    )
    data_interp_spr = np.vstack(interpolated_days)
    anomaly_interp_spr = data_interp_spr - climatology_obs_grid # mean ~ 0
    
    return data_interp_spr, anomaly_interp_spr


# ============================================================
# PRECIPITATION
# ============================================================
def spr_interp_anomaly_precip_var_em(anomaly_rcm_grid, anomaly_obs_grid, climatology_obs_grid,
                                     n_spatterns=10, n_jobs=-1, lambda_var=0.001,
                                     max_iter=20, tol=1e-4, init_A='zero',
                                     ridge_var=False, gamma_ridge_var=0,
                                     ridge_betas=False, gamma_ridge_betas=0,
                                     verbose=True):
    """
    EM optimization for precipitation (multiplicative anomalies).
    Anomalies are centered (mean ~ 0). The model does not center them during the optimization.
    lambda_var = 0 correspond to no VAR(1) smoothing
    """
    
    # Convert inputs
    anomaly_rcm_grid = anomaly_rcm_grid.values if isinstance(anomaly_rcm_grid, pd.DataFrame) else anomaly_rcm_grid
    anomaly_obs_grid = anomaly_obs_grid.values if isinstance(anomaly_obs_grid, pd.DataFrame) else anomaly_obs_grid
    climatology_obs_grid = climatology_obs_grid.values if isinstance(climatology_obs_grid, pd.DataFrame) else climatology_obs_grid
    
    # SVD with centering
    scaler = StandardScaler(with_mean=True, with_std=False)
    anomaly_rcm_grid_scaled = scaler.fit_transform(anomaly_rcm_grid)
    centres_pca = scaler.mean_
    svd_rcm = TruncatedSVD(n_components=n_spatterns, random_state=0)
    svd_rcm.fit(anomaly_rcm_grid_scaled)
    spatterns = svd_rcm.components_.T
    
    # Remove missing columns
    first_day = anomaly_obs_grid[0, :]
    indexes_NA_col = np.where(np.isnan(first_day))[0]
    data_valid = np.delete(anomaly_obs_grid, indexes_NA_col, axis=1)
    spatterns_model = np.delete(spatterns, indexes_NA_col, axis=0)
    
    T = anomaly_obs_grid.shape[0]
    K = n_spatterns
    V = spatterns_model
    
    # Check if VAR component is needed
    use_var = lambda_var >= 1e-8
    if verbose:
        if use_var:
            print(f"Using VAR(1) temporal component (λ={lambda_var:.6f})")
        else:
            print(f"λ={lambda_var:.6f} ≈ 0 → No VAR component (independent temporal estimation)")
    
    # Initialize A
    if use_var:
        if init_A == 'zero':
            A_est = np.zeros((K, K))
        elif init_A == 'identity':
            A_est = np.eye(K)
        else:
            raise ValueError("init_A must be 'zero' or 'identity'")
        
        if verbose:
            print(f"Initialized A with {init_A} matrix")
    else:
        # No VAR: A is not needed
        A_est = np.zeros((K, K))
        rho = 0.0
    
    # Cost function
    def compute_cost_precip(B_mat, A_mat, data, V_mat, lam):
        recon_error = 0.0
        for t in range(T):
            y_centered = data[t, :]  # Deja centré. 
            y_hat_centered = V_mat @ B_mat[t, :]
            recon_error += np.sum((y_centered - y_hat_centered) ** 2)
        
        temp_penalty = 0.0
        if lam >= 1e-8:
            for t in range(1, T):
                residual = B_mat[t, :] - A_mat @ B_mat[t-1, :]
                temp_penalty += np.sum(residual ** 2)
        
        return recon_error + lam * temp_penalty
    
    costs = []
    B = None
    
    # EM iterations
    for iteration in range(max_iter):
        # STEP I: Estimate B with fixed A
        B_new = np.zeros((T, K))
        
        # First time step
        y_0_centered = data_valid[0, :] # Deja centré.
        if ridge_betas:
            G_0 = V.T @ V + gamma_ridge_betas * np.eye(K)
        else:
            G_0 = V.T @ V + 1e-4 * np.eye(K)
        rhs_0 = V.T @ y_0_centered
        B_new[0, :] = np.linalg.solve(G_0, rhs_0)
        
        # Subsequent time steps
        for t in range(1, T):
            y_centered = data_valid[t, :] # Deja centré
            
            if use_var:
                if ridge_betas:
                    G = V.T @ V + lambda_var * np.eye(K) + gamma_ridge_betas * np.eye(K)
                else:
                    G = V.T @ V + lambda_var * np.eye(K) + 1e-4 * np.eye(K) # numerical stability
                rhs = V.T @ y_centered + lambda_var * (A_est @ B_new[t-1, :])
            else:
                # No VAR: each time step is independent
                if ridge_betas:
                    G = V.T @ V + gamma_ridge_betas * np.eye(K)
                else:
                    G = V.T @ V + 1e-4 * np.eye(K) # numerical stability
                rhs = V.T @ y_centered
            
            B_new[t, :] = np.linalg.solve(G, rhs)
        
        B = B_new
        
        # STEP II: Estimate A with fixed B
        if use_var:
            B_past = B[:-1, :]
            B_future = B[1:, :]
            Gram = B_past.T @ B_past
            
            if ridge_var:
                Gram_reg = Gram + gamma_ridge_var * np.eye(K)
            else:
                Gram_reg = Gram + 1e-4 * np.eye(K) # numerical stability
            
            try:
                Gram_inv = np.linalg.inv(Gram_reg)
            except np.linalg.LinAlgError:
                Gram_inv = np.linalg.pinv(Gram_reg)
            
            A_est = B_future.T @ B_past @ Gram_inv
            
            # Check for NaN/Inf in A_est before computing eigenvalues
            if not np.isfinite(A_est).all():
                raise ValueError(
                    f"A_est contains NaN or Inf values. This typically indicates numerical instability. "
                    f"Try increasing lambda_var (current: {lambda_var}) or reducing n_spatterns (current: {n_spatterns})."
                )
            
            # Stability constraint
            rho = np.max(np.abs(np.linalg.eigvals(A_est)))
            if rho > 1:
                A_est = A_est / (rho + 1e-4)
        # else: A_est and rho already set above
        
        # Check convergence
        current_cost = compute_cost_precip(B, A_est, data_valid, V, lambda_var)
        costs.append(current_cost)
        
        if iteration > 0:
            cost_change = abs(current_cost - prev_cost) / (prev_cost + 1e-12)
            if verbose:
                if use_var:
                    print(f"Iter {iteration+1}/{max_iter}: Cost = {current_cost:.6f}, "
                          f"ΔCost = {cost_change:.6e}, ρ(A) = {rho:.4f}")
                else:
                    print(f"Iter {iteration+1}/{max_iter}: Cost = {current_cost:.6f}, "
                          f"ΔCost = {cost_change:.6e}")
            if cost_change < tol:
                if verbose:
                    print(f"✓ Converged after {iteration+1} iterations")
                break
        else:
            if verbose:
                print(f"Iter {iteration+1}/{max_iter}: Cost = {current_cost:.6f}")
        
        prev_cost = current_cost
    
    # Reconstruction
    def reconstruct_day(t):
        beta_t = B[t, :].reshape(-1, 1)
        anomaly_centered = spatterns @ beta_t
        rec = (anomaly_centered + 1) * climatology_obs_grid[t, :].reshape(-1, 1)
        # rec[rec < 0.5] = 0
        return rec.ravel()
    
    interpolated_days = Parallel(n_jobs=n_jobs)(
        delayed(reconstruct_day)(t) for t in tqdm(range(T), desc="Reconstructing")
    )
    data_interp_spr = np.vstack(interpolated_days)
    anomaly_interp_spr = data_interp_spr / climatology_obs_grid # mean ~ 1
    
    return data_interp_spr, anomaly_interp_spr

# Interpolation with VAR

## Precipitation interpolation with SPR_ano_var

In [12]:
# Remove test columns from anomaly_pr_obs
data_pr_train = np.delete(anomaly_pr_obs, test_indexes, axis=1)

# Compute n_pr
n_pr = min(anomaly_pr_grid.shape[1], data_pr_train.shape[1])

# # Define proportion of available spatial patterns to use for the regression as predictors
prop_NA_pr = 0.3

# # Compute number of spatial patterns
n_spatterns_pr = custom_round(n_pr * prop_NA_pr)

# # Define series of lambda_var values
lambda_var_pr = 0.001


# Interpolation with selected hyperparameters
spr_ano_var_pr_A1_10, anomaly_ano_var_pr_A1_10 = spr_interp_anomaly_precip_var_em(
    anomaly_rcm_grid=anomaly_pr_grid_np,
    anomaly_obs_grid=anomaly_pr_obs_np,
    climatology_obs_grid=clim_pr_obs_np,
    #test_index=test_indexes,
    n_spatterns=n_spatterns_pr,
    lambda_var=lambda_var_pr,
    max_iter=20,
    tol=1e-4,
    init_A='zero',
    verbose=False
)

# # Save results as pd.DataFrame and CSV files
spr_ano_var_pr_A1_10 = pd.DataFrame(spr_ano_var_pr_A1_10, columns=data_pr_obs.columns, index=data_pr_obs.index)
spr_ano_var_pr_A1_10.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Interpolation/Interp_A1/Simulations/spr_ano_var_pr_A1_10.csv")
# beta_ano_var_pr_A1_10 = pd.DataFrame(beta_ano_var_pr_A1_10)
# beta_ano_var_pr_A1_10.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data_interp/SPR_var_10/beta_ano_var_pr_A1_10.csv")
# anomaly_ano_var_pr_A1_10 = pd.DataFrame(anomaly_ano_var_pr_A1_10, columns=data_pr_obs.columns, index=data_pr_obs.index)
# anomaly_ano_var_pr_A1_10.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data_interp/SPR_var_10/anomaly_ano_var_pr_A1_10.csv")

Reconstructing: 100%|██████████| 1825/1825 [00:00<00:00, 5211.00it/s]


## Minimum temperature interpolation with SPR_ano_var

In [13]:
# Remove test columns from anomaly_tasmin_obs
data_tasmin_train = np.delete(anomaly_tasmin_obs, test_indexes, axis=1)

# Compute n_tasmin
n_tmin = min(anomaly_tasmin_grid.shape[1], data_tasmin_train.shape[1])

# # Define proportion of available spatial patterns to use for the regression as predictors
prop_NA_tmin = 0.25#0.2 #0.125

# # Compute number of spatial patterns
n_spatterns_tmin = custom_round(n_tmin * prop_NA_tmin)

# # Define series of lambda_var values
lambda_var_tmin = 0.001


# Interpolation with selected hyperparameters
spr_ano_var_tmin_A1_10, anomaly_ano_var_tmin_A1_10 = spr_interp_anomaly_temp_var_em(
    anomaly_rcm_grid=anomaly_tasmin_grid_np,
    anomaly_obs_grid=anomaly_tasmin_obs_np,
    climatology_obs_grid=clim_tasmin_obs_np,
    #test_index=test_indexes,
    n_spatterns=n_spatterns_tmin,
    lambda_var=lambda_var_tmin,
    max_iter=20,
    tol=1e-4,
    init_A='zero',
    verbose=False
)

# # Save results as pd.DataFrame and CSV files
spr_ano_var_tmin_A1_10 = pd.DataFrame(spr_ano_var_tmin_A1_10, columns=data_tasmin_obs.columns, index=data_tasmin_obs.index)
spr_ano_var_tmin_A1_10.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Interpolation/Interp_A1/Simulations/spr_ano_var_tmin_A1_10.csv")
# beta_ano_var_tmin_A1_10 = pd.DataFrame(beta_ano_var_tmin_A1_10)
# beta_ano_var_tmin_A1_10.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data_interp/SPR_var_10/beta_ano_var_tmin_A1_10.csv")
# anomaly_ano_var_tmin_A1_10 = pd.DataFrame(anomaly_ano_var_tmin_A1_10, columns=data_tasmin_obs.columns, index=data_tasmin_obs.index)
# anomaly_ano_var_tmin_A1_10.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data_interp/SPR_var_10/anomaly_ano_var_tmin_A1_10.csv")

Reconstructing: 100%|██████████| 1825/1825 [00:00<00:00, 15129.20it/s]


## Maximum temperature interpolation with SPR_ano_var

In [14]:
# Remove test columns from anomaly_tasmax_obs
data_tasmax_train = np.delete(anomaly_tasmax_obs, test_indexes, axis=1)

# Compute n_tasmin
n_tasmax = min(anomaly_tasmax_grid.shape[1], data_tasmax_train.shape[1])

# Selected K, the proportion of available spatial patterns to use for the regression as predictors
prop_NA_tmax = 0.25

## Compute number of spatial patterns
n_spatterns_tmax = custom_round(n_tasmax * prop_NA_tmax)

# Selected lambda
lambda_var_tmax = 0.001

# Interpolation with selected hyperparameters
spr_ano_var_tmax_A1_10, anomaly_ano_var_tmax_A1_10 = spr_interp_anomaly_temp_var_em(
    anomaly_rcm_grid=anomaly_tasmax_grid_np,
    anomaly_obs_grid=anomaly_tasmax_obs_np,
    climatology_obs_grid=clim_tasmax_obs_np,
    #test_index=test_indexes,
    n_spatterns=n_spatterns_tmax,
    lambda_var=lambda_var_tmax,
    max_iter=20,
    tol=1e-4,
    init_A='zero',
    verbose=False
)

# Save results as pd.DataFrame and CSV files
spr_ano_var_tmax_A1_10 = pd.DataFrame(spr_ano_var_tmax_A1_10, columns=data_tasmax_obs.columns, index=data_tasmax_obs.index)
spr_ano_var_tmax_A1_10.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Interpolation/Interp_A1/Simulations/spr_ano_var_tmax_A1_10.csv")
# beta_ano_var_tmax_A1_10 = pd.DataFrame(beta_ano_var_tmax_A1_10)
# beta_ano_var_tmax_A1_10.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data_interp/SPR_var_10/beta_ano_var_tmax_A1_10.csv")
# anomaly_ano_var_tmax_A1_10 = pd.DataFrame(anomaly_ano_var_tmax_A1_10, columns=data_tasmax_obs.columns, index=data_tasmax_obs.index)
# anomaly_ano_var_tmax_A1_10.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data_interp/SPR_var_10/anomaly_ano_var_tmax_A1_10.csv")

Reconstructing: 100%|██████████| 1825/1825 [00:00<00:00, 15049.78it/s]


# Interpolation with SPR_ANO

## PRECIP

In [15]:
# Remove test columns from anomaly_pr_obs
data_pr_train = np.delete(anomaly_pr_obs, test_indexes, axis=1)

# Compute n_pr
n_pr = min(anomaly_pr_grid.shape[1], data_pr_train.shape[1])

# # Define proportion of available spatial patterns to use for the regression as predictors
prop_NA_pr = 0.25

# # Compute number of spatial patterns
n_spatterns_pr = custom_round(n_pr * prop_NA_pr)

# # Define series of lambda_var values
lambda_var_pr = 0


# Interpolation with selected hyperparameters
spr_ano_pr_A1_10, anomaly_ano_pr_A1_10 = spr_interp_anomaly_precip_var_em(
    anomaly_rcm_grid=anomaly_pr_grid_np,
    anomaly_obs_grid=anomaly_pr_obs_np,
    climatology_obs_grid=clim_pr_obs_np,
    #test_index=test_indexes,
    n_spatterns=n_spatterns_pr,
    lambda_var=lambda_var_pr,
    max_iter=20,
    tol=1e-4,
    init_A='zero',
    verbose=False
)

# # Save results as pd.DataFrame and CSV files
spr_ano_pr_A1_10 = pd.DataFrame(spr_ano_pr_A1_10, columns=data_pr_obs.columns, index=data_pr_obs.index)
spr_ano_pr_A1_10.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Interpolation/Interp_A1/Simulations/spr_ano_pr_A1_10.csv")
# beta_ano_pr_A1_10 = pd.DataFrame(beta_ano_pr_A1_10)
# beta_ano_pr_A1_10.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data_interp/SPR_var_10/beta_ano_pr_A1_10.csv")
# anomaly_ano_pr_A1_10 = pd.DataFrame(anomaly_ano_pr_A1_10, columns=data_pr_obs.columns, index=data_pr_obs.index)
# anomaly_ano_var_pr_A1_10.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data_interp/SPR_var_10/anomaly_ano_var_pr_A1_10.csv")

Reconstructing: 100%|██████████| 1825/1825 [00:00<00:00, 17240.37it/s]


## TMIN

In [16]:
# Remove test columns from anomaly_tasmin_obs
data_tasmin_train = np.delete(anomaly_tasmin_obs, test_indexes, axis=1)

# Compute n_tasmin
n_tmin = min(anomaly_tasmin_grid.shape[1], data_tasmin_train.shape[1])

# # Define proportion of available spatial patterns to use for the regression as predictors
prop_NA_tmin = 0.25#0.2 #0.125

# # Compute number of spatial patterns
n_spatterns_tmin = custom_round(n_tmin * prop_NA_tmin)

# # Define series of lambda_var values
lambda_var_tmin = 0


# Interpolation with selected hyperparameters
spr_ano_tmin_A1_10, anomaly_ano_tmin_A1_10 = spr_interp_anomaly_temp_var_em(
    anomaly_rcm_grid=anomaly_tasmin_grid_np,
    anomaly_obs_grid=anomaly_tasmin_obs_np,
    climatology_obs_grid=clim_tasmin_obs_np,
    #test_index=test_indexes,
    n_spatterns=n_spatterns_tmin,
    lambda_var=lambda_var_tmin,
    max_iter=20,
    tol=1e-4,
    init_A='zero',
    verbose=False
)

# # Save results as pd.DataFrame and CSV files
spr_ano_tmin_A1_10 = pd.DataFrame(spr_ano_tmin_A1_10, columns=data_tasmin_obs.columns, index=data_tasmin_obs.index)
spr_ano_tmin_A1_10.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Interpolation/Interp_A1/Simulations/spr_ano_tmin_A1_10.csv")
# beta_ano_tmin_A1_10 = pd.DataFrame(beta_ano_tmin_A1_10)
# beta_ano_tmin_A1_10.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data_interp/SPR_ano_10/beta_ano_tmin_A1_10.csv")
# anomaly_ano_tmin_A1_10 = pd.DataFrame(anomaly_ano_tmin_A1_10, columns=data_tasmin_obs.columns, index=data_tasmin_obs.index)
# anomaly_ano_tmin_A1_10.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data_interp/SPR_ano_10/anomaly_ano_tmin_A1_10.csv")

Reconstructing: 100%|██████████| 1825/1825 [00:00<00:00, 15690.55it/s]


## TMAX

In [17]:
# Remove test columns from anomaly_tasmax_obs
data_tasmax_train = np.delete(anomaly_tasmax_obs, test_indexes, axis=1)

# Compute n_tasmin
n_tasmax = min(anomaly_tasmax_grid.shape[1], data_tasmax_train.shape[1])

# Selected K, the proportion of available spatial patterns to use for the regression as predictors
prop_NA_tmax = 0.25

## Compute number of spatial patterns
n_spatterns_tmax = custom_round(n_tasmax * prop_NA_tmax)

# Selected lambda
lambda_var_tmax = 0

# Interpolation with selected hyperparameters
spr_ano_tmax_A1_10, anomaly_ano_tmax_A1_10 = spr_interp_anomaly_temp_var_em(
    anomaly_rcm_grid=anomaly_tasmax_grid_np,
    anomaly_obs_grid=anomaly_tasmax_obs_np,
    climatology_obs_grid=clim_tasmax_obs_np,
    #test_index=test_indexes,
    n_spatterns=n_spatterns_tmax,
    lambda_var=lambda_var_tmax,
    max_iter=20,
    tol=1e-4,
    init_A='zero',
    verbose=False
)

# Save results as pd.DataFrame and CSV files
spr_ano_tmax_A1_10 = pd.DataFrame(spr_ano_tmax_A1_10, columns=data_tasmax_obs.columns, index=data_tasmax_obs.index)
spr_ano_tmax_A1_10.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Interpolation/Interp_A1/Simulations/spr_ano_tmax_A1_10.csv")
# beta_ano_tmax_A1_10 = pd.DataFrame(beta_ano_tmax_A1_10)
# beta_ano_tmax_A1_10.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data_interp/SPR_ano_10/beta_ano_tmax_A1_10.csv")
# anomaly_ano_tmax_A1_10 = pd.DataFrame(anomaly_ano_tmax_A1_10, columns=data_tasmax_obs.columns, index=data_tasmax_obs.index)
# anomaly_ano_tmax_A1_10.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data_interp/SPR_ano_10/anomaly_ano_tmax_A1_10.csv")

Reconstructing: 100%|██████████| 1825/1825 [00:00<00:00, 16004.71it/s]
